In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

# 4 features for each sample
X = torch.tensor([
    [1.0, 2.0, 1.0, 2.0],
    [1.5, 1.8, 1.2, 2.1],
    [2.0, 1.5, 1.0, 1.8],
    [7.0, 8.0, 7.5, 8.5],
    [8.0, 7.5, 8.0, 7.0],
    [7.5, 8.5, 7.0, 8.0]
])

# 0 = Healthy
# 1 = Disease
y = torch.tensor([0, 0, 0, 1, 1, 1])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: torch.Size([6, 4])
y shape: torch.Size([6])


In [2]:
# Build first multilayer network

model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),

    nn.Linear(16, 8),
    nn.ReLU(),

    nn.Linear(8, 2)
)

print(model)

Sequential(
  (0): Linear(in_features=4, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=2, bias=True)
)


In [3]:
# Look at the logits before training

logits = model(X)

print(logits)
print(logits.shape)



tensor([[ 0.4911, -0.1685],
        [ 0.5082, -0.1713],
        [ 0.5249, -0.1782],
        [ 1.5007,  0.1604],
        [ 1.6301,  0.1033],
        [ 1.7406,  0.1228]], grad_fn=<AddmmBackward0>)
torch.Size([6, 2])


In [5]:
# Define the loss

loss_fn = nn.CrossEntropyLoss()

print(loss_fn)

CrossEntropyLoss()


In [6]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [7]:
# Train the model
for epoch in range(200):

    # 1. Forward pass
    logits = model(X)

    # 2. Calculate loss
    loss = loss_fn(logits, y)

    # 3. Clear old gradients
    optimizer.zero_grad()

    # 4. Calculate new gradients
    loss.backward()

    # 5. Update weights and biases
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.0540
Epoch 20, Loss: 0.3011
Epoch 40, Loss: 0.1092
Epoch 60, Loss: 0.0135
Epoch 80, Loss: 0.0028
Epoch 100, Loss: 0.0014
Epoch 120, Loss: 0.0009
Epoch 140, Loss: 0.0007
Epoch 160, Loss: 0.0005
Epoch 180, Loss: 0.0004


In [8]:
# Model for prediction

model.eval()

with torch.no_grad():
    logits = model(X)
    predictions = torch.argmax(logits, dim=1)

print("Predictions:", predictions)
print("Actual:     ", y)

Predictions: tensor([0, 0, 0, 1, 1, 1])
Actual:      tensor([0, 0, 0, 1, 1, 1])


In [9]:
# Evaluate performance

accuracy = (predictions == y).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 1.0


In [10]:
# Test dataset

X_test = torch.tensor([
    [1.3, 2.1, 1.4, 1.7],  # should be Healthy
    [7.8, 7.2, 8.1, 7.6]   # should be Disease
])

model.eval()

with torch.no_grad():
    test_logits = model(X_test)
    test_predictions = torch.argmax(test_logits, dim=1)

print("Logits:")
print(test_logits)

print("Predictions:", test_predictions)

Logits:
tensor([[ 3.7841, -3.2805],
        [-5.2746,  6.0402]])
Predictions: tensor([0, 1])


In [11]:
# Look for the probabilities

probabilities = torch.softmax(test_logits, dim=1)

print(probabilities)


tensor([[9.9915e-01, 8.5406e-04],
        [1.2190e-05, 9.9999e-01]])
